In [1]:
!pip install fairlearn pgmpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.0/240.0 kB 4.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 29.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 74.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 60.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 26.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 4.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import random
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination

# ---------------- reproducibility ----------------
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

In [3]:
def preprocess_adult_for_fair_bbn(csv_path='/kaggle/input/adult-census-income/adult.csv'):
    #removes missing values, income categorical
    df = pd.read_csv(csv_path)
    df = df[~df.isin(['?']).any(axis=1)].reset_index(drop=True)
    df['income'] = np.where(df['income']=='>50K', 1, 0)
    # Upsample positives (preserve race/sex proportion)
    #postive income count
    more_50k = df[df['income']==1]
    #group positive income by race and sex and store them with their count
    dist = more_50k.groupby(['race','sex']).size().reset_index(name='count')
    # 22654 is our target we want after upsampling (must be the number of 0s in the dataset)
    #we want the proportion. like if white male have 5000 count out of all positivies(say 10K),
    #then we have the proportional upcount with the below step
    dist['up_count'] = (dist['count']*22654/dist['count'].sum()).round().astype(int)
    ups = []
    #iterates over each row and then resample with replacement acc to the proportion calculated
    # concat that with the 0 income ones and frac = 1 shuffles the df
    for _, row in dist.iterrows():
        group = more_50k[(more_50k['race']==row['race']) & (more_50k['sex']==row['sex'])]
        ups.append(resample(group, replace=True, n_samples=row['up_count'], random_state=seed))
    df = pd.concat([df[df['income']==0]] + ups).sample(frac=1, random_state=seed).reset_index(drop=True)
    # Sensitive labels - mapped for Adversarial purposes
    race_map = {"White": 0,"Black": 1,"Asian-Pac-Islander": 2,"Amer-Indian-Eskimo": 3,"Other": 4}
    sex_map = {"Male": 0,"Female": 1}
    df['race_label'] = df['race'].map(race_map)
    df['sex_label'] = df['sex'].map(sex_map)
    # Columns
    categorical_cols = ['workclass','education','marital.status','occupation','relationship','native.country']
    numeric_cols = ['age','fnlwgt','education.num','capital.gain','capital.loss','hours.per.week']
    #SEPERATELY PROCESSING FOR NN AND BBN
    #we one hot encode for nn but only map for bbn. standard scalar for nn but binning for bbn
    # Preprocessor for NN
    preproc = ColumnTransformer([
        ('num', Pipeline([('scaler', StandardScaler())]), numeric_cols),
        ('cat', Pipeline([('ohe', OneHotEncoder(handle_unknown='ignore'))]), categorical_cols)
    ])
    # BBN: discretize numeric, integer codes for categorical + sensitive
    #cause BBN needs categorical values only
    bbn_df = df.copy()
    for col in numeric_cols:
        bbn_df[col] = pd.qcut(bbn_df[col], 5, labels=False, duplicates='drop').astype(int)
    for col in categorical_cols + ['race','sex']:
        bbn_df[col] = bbn_df[col].astype('category').cat.codes.astype(int)
    # Split train/test
    X = df.drop(columns=['income','race_label','sex_label'])
    y = df['income'].values
    sens_race = df['race_label']
    sens_sex = df['sex_label']
    X_train_raw, X_test_raw, y_train, y_test, sens_race_train, sens_race_test, sens_sex_train, sens_sex_test = \
        train_test_split(X, y, sens_race, sens_sex, test_size=0.2, stratify=y, random_state=seed)
    # NN preprocessing
    X_train_nn = preproc.fit_transform(X_train_raw)
    X_test_nn = preproc.transform(X_test_raw)
    # BBN datasets aligned
    bbn_train = bbn_df.loc[X_train_raw.index].reset_index(drop=True)
    bbn_test = bbn_df.loc[X_test_raw.index].reset_index(drop=True)
    return {
        'X_train_nn': X_train_nn, 'X_test_nn': X_test_nn,
        'bbn_train_df': bbn_train, 'bbn_test_df': bbn_test,
        'y_train': y_train, 'y_test': y_test,
        'sens_race_train': sens_race_train.reset_index(drop=True),
        'sens_race_test': sens_race_test.reset_index(drop=True),
        'sens_sex_train': sens_sex_train.reset_index(drop=True),
        'sens_sex_test': sens_sex_test.reset_index(drop=True)
    }


**makes a BBN with features to target, fits them and queries them for income = 1. this gives the relationship bw the features and the target that acts as a prior for the next steps.******

In [4]:
# ------------------------ BBN ------------------------
class SimpleFairBBN:
    #list of features to include in the bbn and the target attribute
    def __init__(self, feature_names, target_name='target'):
        self.feature_names = feature_names
        self.target_name = target_name
        self.model = None #will hold DiscreteBayesianNetwork
        self.inference = None #will hold VariableElimination object for querying probabs
    #df_discrete is the dataset with discretised features and y is target labels.
    def fit(self, df_discrete, y, include_sensitive=True):
        #combine feature and target for BBN 
        data = df_discrete.copy().reset_index(drop=True)
        data[self.target_name] = y
        # Use top 6 features + optionally sensitive attrs
        candidate_features = list(df_discrete.columns)
        selected = candidate_features[:6]
        if include_sensitive: #this is optional, can mess around with this
            if 'race' in df_discrete.columns: selected.append('race')
            if 'sex' in df_discrete.columns: selected.append('sex')
        edges = [(f, self.target_name) for f in selected] #feature to target edge
        #fitting the BBN
        self.feature_names = selected
        self.model = DiscreteBayesianNetwork(edges)
        # BDeu estimator with small equivalent sample size for fairness smoothing
        self.model.fit(data, estimator=BayesianEstimator, prior_type='BDeu', equivalent_sample_size=5)
        self.inference = VariableElimination(self.model)

    def predict_proba(self, df_discrete):
        Xdf = df_discrete.reset_index(drop=True)
        probs = [] #we predict probability of where income = 1
        for _, row in Xdf.iterrows():
            evidence = {} #features that will be given to the BBN as observations
            used = 0 #this limits the no of features used as evidence 
            for f in self.feature_names:
                if f in row and not pd.isna(row[f]) and used<3: #make the feature isnt nan and used less than 3 times
                    try: evidence[f] = int(row[f]); used+=1 #convert feature val to int
                    except: continue
            try:
                if evidence: #query for inference if evidence exists
                    q = self.inference.query(variables=[self.target_name], evidence=evidence)
                else: #else marginal probability
                    q = self.inference.query(variables=[self.target_name])
                p1 = q.values[1] if len(q.values)==2 else 0.5 #q.values[1] gives probab of 1 
                #default to 0.5 if something went wrong and the values are not binary
            except: p1=0.5
            probs.append(p1) #p1 is the predicted probab of 1 row
        return np.array(probs)



**adapter -> convert the priors of the BBN to actual target and also hidden rep.This hidden rep is used by the adversary to predict the sex and race attrs. if it succeeds, the adapter is penalized.**


        +----------------------+
        |  Discrete BBN        |
        | (SimpleFairBBN)      |
        |                      |
        | Inputs: discretized  |
        | features (+race/sex) |
        +----------+-----------+
                   |
      Predicts probabilities:
      P(y|all), P(y|race), P(y|sex)
                   v
        +----------------------+
        |     AdapterNN        |
        |  (in_dim=3)         |
        +----------------------+
        | fc1: 3 -> 32        |
        | ReLU                |
        | fc2: 32 -> 16       |
        | ReLU                |
        | out: 16 -> 1        | <-- Logit (raw score for target)
        +----------+-----------+
                   |
          Hidden features h2 (16) ───────┐
                   |                     |
                   v                     |
        Sigmoid → P(y=1) (probability)   |
                                         |
                                +--------v---------+
                                |   AdversaryNN    |
                                +-----------------+
                                | shared: 16 ->32 |
                                | ReLU            |
                                +--------+--------+
                                         |
                          -------------------------------
                          |                             |
                    race_head: 32 -> num_race_classes   sex_head: 32 -> num_sex_classes
                    logits for race prediction          logits for sex prediction
****

In [5]:
# ---------------- Adapter + Adversary ----------------
#actual predictor which ips the vals of the bbn
class AdapterNN(nn.Module):
    def __init__(self, in_dim=3, hidden_dim=32):  # 3 BBN marginals as input - [P(y|all), P(y|race), P(y|sex)]
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, hidden_dim//2)
        self.out = nn.Linear(hidden_dim//2,1) # 1 because binary classification 
    def forward(self, x, return_features=False):
        h = self.act(self.fc1(x))
        h2 = self.act(self.fc2(h))
        logit = self.out(h2)
        if return_features: 
            return logit,h2 #logit is wht we get raw before applying sigmoid 
        return logit
#mitigation and predicts sensitive attr from the adapters rep 
class AdversaryNN(nn.Module):
    def __init__(self, in_dim, race_classes, sex_classes): #ip is hidden dim of the adapter
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(in_dim,32), nn.ReLU())
        self.race_head = nn.Linear(32, race_classes) #diff predictors for race and sex
        self.sex_head = nn.Linear(32, sex_classes)
    def forward(self,x):
        s=self.shared(x)
        return self.race_head(s), self.sex_head(s) #returns logits for sex and race 


base model has been changed to NN

In [6]:
# ---------------- Training ----------------
#hyperparams - data_dict - full data, lamda_adv - weight for adv loss/ penalization
# alpha_bbn - weight for bbn fairness regualation 
def train_fair_bbn_adapter(data_dict, lambda_adv=0.5, alpha_bbn=0.5, epochs=60, batch_size=256, lr=1e-3):
    #extra all the req data 
    X_train_nn = data_dict['X_train_nn']; X_test_nn = data_dict['X_test_nn']
    bbn_train_df = data_dict['bbn_train_df']; bbn_test_df = data_dict['bbn_test_df']
    y_train = data_dict['y_train']; y_test = data_dict['y_test']
    sens_race_train = data_dict['sens_race_train']; sens_race_test = data_dict['sens_race_test']
    sens_sex_train = data_dict['sens_sex_train']; sens_sex_test = data_dict['sens_sex_test']
    
    # ---------------- Baseline NN ---------------- (this is the base model and its params)
    print("Training baseline MLP...")
    baseline = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=300, random_state=seed)
    baseline.fit(X_train_nn, y_train)
    base_pred = baseline.predict(X_test_nn)
    base_acc = accuracy_score(y_test, base_pred)
    base_race_dp = demographic_parity_difference(y_test, base_pred, sensitive_features=sens_race_test)
    base_race_eo = equalized_odds_difference(y_test, base_pred, sensitive_features=sens_race_test)
    base_sex_dp = demographic_parity_difference(y_test, base_pred, sensitive_features=sens_sex_test)
    base_sex_eo = equalized_odds_difference(y_test, base_pred, sensitive_features=sens_sex_test)
    print(f"Baseline MLP Accuracy: {base_acc:.4f}")

    # ---------------- Fair BBN ----------------
    print("Training fair BBN...")
    bbn = SimpleFairBBN(feature_names=list(bbn_train_df.columns))
    bbn.fit(bbn_train_df, y_train, include_sensitive=True)

    # Generate multiple BBN marginals as input features: P(y|features), P(y|race), P(y|sex)
    p_all = bbn.predict_proba(bbn_train_df).reshape(-1,1)
    p_race = bbn.predict_proba(bbn_train_df[['race']]).reshape(-1,1)
    p_sex = bbn.predict_proba(bbn_train_df[['sex']]).reshape(-1,1)
    #stack all the prredicted probab horizontally
    adapter_in_train = torch.tensor(np.hstack([p_all,p_race,p_sex]), dtype=torch.float32)

    p_all_test = bbn.predict_proba(bbn_test_df).reshape(-1,1)
    p_race_test = bbn.predict_proba(bbn_test_df[['race']]).reshape(-1,1)
    p_sex_test = bbn.predict_proba(bbn_test_df[['sex']]).reshape(-1,1)
    adapter_in_test = torch.tensor(np.hstack([p_all_test,p_race_test,p_sex_test]), dtype=torch.float32)

    #convert target and sensitive attributes into tensors
    y_train_t = torch.tensor(y_train.reshape(-1,1), dtype=torch.float32)
    y_test_t  = torch.tensor(y_test.reshape(-1,1), dtype=torch.float32)
    race_train_t = torch.tensor(sens_race_train.values.astype(int), dtype=torch.long)
    sex_train_t  = torch.tensor(sens_sex_train.values.astype(int), dtype=torch.long)
    #use dataloader for batch training and shuffling 
    train_loader = DataLoader(TensorDataset(adapter_in_train, y_train_t, race_train_t, sex_train_t), 
                              batch_size=batch_size, shuffle=True)
    #iinitialize adapter and adversary, adam optimizers, 
    #losses -  BCEWithLogitsLoss → binary classification for adapter, CrossEntropyLoss → multi-class classification for adversary
    adapter = AdapterNN(in_dim=3, hidden_dim=64)
    adversary = AdversaryNN(in_dim=32, race_classes=len(sens_race_train.unique()), sex_classes=len(sens_sex_train.unique()))
    adapter_opt = optim.Adam(adapter.parameters(), lr=lr)
    adv_opt = optim.Adam(adversary.parameters(), lr=lr)
    pred_loss_fn = nn.BCEWithLogitsLoss()
    adv_loss_fn = nn.CrossEntropyLoss()

    # Training loop
    print("Training adapter with adversarial + BBN fairness...")
    for epoch in range(epochs):
        adapter.train(); adversary.train()
        total_adapter_loss=0.0; total_adv_loss=0.0
        for batch in train_loader:
            batch_in, batch_y, batch_race, batch_sex = batch

            # --- Train adversary ---
            with torch.no_grad():
                _, features = adapter(batch_in, return_features=True)
            adv_opt.zero_grad()
            r_logits, s_logits = adversary(features)
            #loss of adv includes tha of both race and sex avg
            adv_loss = (adv_loss_fn(r_logits,batch_race) + adv_loss_fn(s_logits,batch_sex))/2.0
            adv_loss.backward(); adv_opt.step(); total_adv_loss+=adv_loss.item()

            # --- Train adapter ---
            adapter_opt.zero_grad()
            logit, features = adapter(batch_in, return_features=True)
            pred_loss = pred_loss_fn(logit,batch_y)
            r_logits2, s_logits2 = adversary(features)
            adv_loss_for_adapter = (adv_loss_fn(r_logits2,batch_race) + adv_loss_fn(s_logits2,batch_sex))/2.0

            # BBN fairness regularization: DP between race groups
            dp_penalty = torch.abs(features[batch_race==0].mean() - features[batch_race==1].mean())
            #MAIN LOSS CALCULATOR, add penalty and subtract pred(to minimize) - adv loss (want to fool adv)
            total_loss = pred_loss - lambda_adv*adv_loss_for_adapter + alpha_bbn*dp_penalty
            total_loss.backward()
            adapter_opt.step()
            total_adapter_loss += total_loss.item()

        if epoch % 10==0 or epoch==epochs-1:
            print(f"Epoch {epoch:3d} | Adv Loss: {total_adv_loss/len(train_loader):.4f} | Adapter Loss: {total_adapter_loss/len(train_loader):.4f}")

    # ---------------- Evaluation ----------------
    adapter.eval(); adversary.eval()
    with torch.no_grad():
        test_logit,_ = adapter(adapter_in_test, return_features=True)
        test_probs = torch.sigmoid(test_logit.cpu()).numpy().flatten()
        test_pred = (test_probs>0.5).astype(int)

    adv_acc = accuracy_score(y_test, test_pred)
    adv_race_dp = demographic_parity_difference(y_test, test_pred, sensitive_features=sens_race_test)
    adv_race_eo = equalized_odds_difference(y_test, test_pred, sensitive_features=sens_race_test)
    adv_sex_dp = demographic_parity_difference(y_test, test_pred, sensitive_features=sens_sex_test)
    adv_sex_eo = equalized_odds_difference(y_test, test_pred, sensitive_features=sens_sex_test)

    # Print comparison
    print("\nBASELINE MLP RESULTS:")
    print(f" Accuracy: {base_acc:.4f}")
    print(f" Race DP: {base_race_dp:.4f}, Race EO: {base_race_eo:.4f}")
    print(f" Sex  DP: {base_sex_dp:.4f}, Sex  EO: {base_sex_eo:.4f}")

    print("\nBBN + Adapter (Adversarial + Fairness) RESULTS:")
    print(f" Accuracy: {adv_acc:.4f}")
    print(f" Race DP: {adv_race_dp:.4f}, Race EO: {adv_race_eo:.4f}")
    print(f" Sex  DP: {adv_sex_dp:.4f}, Sex  EO: {adv_sex_eo:.4f}")

    return {'baseline':{'pred':base_pred,'acc':base_acc,'race_dp':base_race_dp,'race_eo':base_race_eo,
                        'sex_dp':base_sex_dp,'sex_eo':base_sex_eo},
            'bbn_adapter':{'pred':test_pred,'acc':adv_acc,'race_dp':adv_race_dp,'race_eo':adv_race_eo,
                           'sex_dp':adv_sex_dp,'sex_eo':adv_sex_eo}}



In [7]:
# ---------------- Run ----------------
if __name__=='__main__':
    print("Preprocessing dataset...")
    data_dict = preprocess_adult_for_fair_bbn()
    results = train_fair_bbn_adapter(data_dict, lambda_adv=0.5, alpha_bbn=0.5, epochs=60)


Preprocessing dataset...
Training baseline MLP...
Baseline MLP Accuracy: 0.8746
Training fair BBN...
Training adapter with adversarial + BBN fairness...
Epoch   0 | Adv Loss: 0.9658 | Adapter Loss: 0.2124
Epoch  10 | Adv Loss: 0.5353 | Adapter Loss: 0.4255
Epoch  20 | Adv Loss: 0.5354 | Adapter Loss: 0.4255
Epoch  30 | Adv Loss: 0.5355 | Adapter Loss: 0.4254
Epoch  40 | Adv Loss: 0.5353 | Adapter Loss: 0.4255
Epoch  50 | Adv Loss: 0.5353 | Adapter Loss: 0.4255
Epoch  59 | Adv Loss: 0.5355 | Adapter Loss: 0.4255

BASELINE MLP RESULTS:
 Accuracy: 0.8746
 Race DP: 0.3269, Race EO: 0.1584
 Sex  DP: 0.3241, Sex  EO: 0.1649

BBN + Adapter (Adversarial + Fairness) RESULTS:
 Accuracy: 0.5000
 Race DP: 0.0000, Race EO: 0.0000
 Sex  DP: 0.0000, Sex  EO: 0.0000
